# Proyecto 1: EDA Honesto, Feature Engineering y Baseline k-NN

**Ciencia de Datos, Sección A** · Segundo Semestre 2026  
**Autor:** Josué Aguilar  
**Dataset elegido:** Censo completo del RMS Titanic (`titanic5.xlsx - Titanic5_all.csv`)  
**Archivo final:** `P1-Aguilar.ipynb`  

---

### Rúbrica resumida

| Criterio | Pts | Qué buscamos |
|---|---|---|
| Pregunta y datos | 2 | Pregunta clara, provenance del dataset |
| EDA | 3 | Hallazgos, no galería de gráficas |
| Features | 2 | Cada decisión justificada, sin leakage |
| Baseline | 2 | Validación correcta, métrica adecuada |
| Límites | 1 | Qué no puede concluirse con estos datos |

## 1. Pregunta y contexto

El naufragio del RMS Titanic en abril de 1912 constituye uno de los eventos marítimos y sociológicos más documentados de la historia moderna. Tradicionalmente, los análisis predictivos sobre este desastre se han restringido a subconjuntos de pasajeros civiles, omitiendo el papel determinante de la tripulación técnica y de servicio, o asumiendo homogeneidad en los protocolos de salvamento. Sin embargo, la evacuación estuvo regida por tensiones estructurales: la directiva marítima de "mujeres y niños primero", la proximidad física a la cubierta superior según la tarifa abonada y la asignación laboral de emergencia de los miembros de la tripulación.

Este análisis busca responder: ¿con qué grado de certeza es posible estimar la probabilidad de supervivencia de un tripulante o pasajero del Titanic utilizando únicamente atributos demográficos, puerto de embarque, estructura familiar y rol funcional a bordo (`Class/Dept`), y en qué medida este modelo refleja la disparidad en la ejecución del protocolo de evacuación entre las distintas clases sociales y departamentos operativos?

Esta respuesta resulta de alto valor para historiadores cuantitativos, sociólogos de catástrofes y analistas de gestión de riesgos y seguridad humana. Permite determinar con rigor empírico si los patrones de supervivencia obedecieron primariamente a normas de caballería socialmente prescritas, a jerarquías socioeconómicas institucionalizadas o al rol logístico asignado durante la emergencia. Si la evidencia demostrara que el rol operativo y la clase social condicionaron la supervivencia por encima de los factores demográficos universales, se confirmaría que las emergencias no anulan las desigualdades preexistentes, sino que las exacerban bajo la apariencia de protocolos neutrales.

**Pregunta central:** ¿Con qué grado de certeza es posible estimar la probabilidad de supervivencia de un tripulante o pasajero del Titanic utilizando únicamente atributos demográficos, puerto de embarque, estructura familiar y rol funcional a bordo (`Class/Dept`), y en qué medida este modelo refleja la disparidad en la ejecución del protocolo de evacuación entre las distintas clases sociales y departamentos operativos?

**Tipo de problema:** Clasificación binaria supervisada.

**Variable objetivo:** `Survived` (0 = No sobrevivió / Fallecido, 1 = Sobrevivió).

## 2. Los datos

- **Fuente:** Registros censales compilados de la *Encyclopedia Titanica* en conjunción con los manifiestos oficiales de embarque de los puertos de Southampton, Cherbourg, Queenstown y Belfast.
- **Archivo analizado:** `titanic5.xlsx - Titanic5_all.csv`.
- **Fecha de descarga:** Septiembre de 2026.
- **Licencia y términos de uso:** Acceso público con fines de docencia, investigación histórica y análisis académico sin fines comerciales.
- **Unidad de observación:** Cada registro individual representa a una persona única que se encontraba formalmente a bordo del RMS Titanic al momento del siniestro (total: 2,208 registros censales completos entre pasajeros de todas las clases y personal de todas las áreas de servicio y oficiales).

### Diccionario preliminar de variables

| Variable | Tipo conceptual | Descripción |
|---|---|---|
| `Name_ID` | Identificador | Clave numérica única de la persona en el registro |
| `Name`, `First`, `Last` | Texto | Nombre completo y apellidos del individuo |
| `Sex`, `Female`, `Male` | Categórica / Binaria | Sexo biológico reportado en el manifiesto |
| `Age`, `Age_F`, `Age_F_Code` | Numérica / Texto | Edad registrada en años y códigos de aproximación |
| `Class/Dept` | Categórica | Rol funcional a bordo (1st, 2nd, 3rd Class, Deck Crew, Engineering Crew, Victualling Crew, Restaurant Staff, Band) |
| `Class` | Categórica / Numérica | Indicador simplificado de clase de pasaje |
| `Ticket` | Alfanumérico | Número de boleto comercial (disponible en pasaje civil) |
| `Price` | Numérico / Texto | Tarifa pagada expresada en formato de libras esterlinas históricas |
| `Joined` | Categórica | Puerto donde abordó el barco (Southampton, Cherbourg, Queenstown, Belfast) |
| `Job`, `Occupation` | Texto | Ocupación laboral o función específica reportada |
| `DoB`, `DoB_Clean`, `Year_Birth` | Fecha / Numérica | Fecha y año de nacimiento |
| `Date_Death` | Fecha | Fecha de fallecimiento registrada |
| `Boat [Body]` | Alfanumérico | Identificador del bote salvavidas de rescate o número de cuerpo recuperado |
| `URL` | Texto | Enlace a la ficha biográfica de Encyclopedia Titanica |
| `sibsp` | Numérica entera | Número de cónyuges, hermanos y hermanas a bordo |
| `parch` | Numérica entera | Número de padres e hijos a bordo |
| `Survived` | Binaria (Target) | Supervivencia al desastre (1 = Sobrevivió, 0 = Falleció) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fijar semilla para reproducibilidad
RANDOM_STATE = 42

# Configuracion estetica para graficas legibles
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
sns.set_theme(style='whitegrid')

# Cargar el dataset censal completo
dataset_path = 'titanic5.xlsx - Titanic5_all.csv'
df = pd.read_csv(dataset_path)

print(f'Dataset cargado exitosamente: {dataset_path}')
print(f'Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas')

Dataset cargado exitosamente: titanic5.xlsx - Titanic5_all.csv
Dimensiones iniciales: 2208 filas y 27 columnas


## 3. Primer barrido

A continuación se realiza una inspección inicial de las dimensiones, tipos de datos, primeros registros y conteos de valores ausentes para obtener un diagnóstico preliminar del censo.

In [2]:
# Inspeccion general: dimensiones, vista inicial y recuento de nulos
print('--- FORMA DEL CONJUNTO DE DATOS ---')
print(f'Filas: {df.shape[0]}, Columnas: {df.shape[1]}\n')

print('--- PRIMERAS CINCO FILAS ---')
display(df.head())

print('\n--- INFORMACION GENERAL DE TIPOS (df.info) ---')
df.info()

print('\n--- RECUENTO DE VALORES NULOS POR VARIABLE ---')
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({
    'Valores Nulos': null_counts,
    'Porcentaje (%)': null_percent.round(2)
})
display(null_summary[null_summary['Valores Nulos'] > 0])

--- FORMA DEL CONJUNTO DE DATOS ---
Filas: 2208, Columnas: 27

--- PRIMERAS CINCO FILAS ---


,Name_ID,Name,Female,Male,Sex,Age,Class/Dept,Class,Ticket,Joined,...,Title,First,DoB,Year_Birth,Date_Death,DoB_Clean,Age_F_Code,Age_F,sibsp,parch
0,531,"DEAN, Miss Elizabeth Gladys 'Millvina'",1,0,female,0.17,3rd Class Passenger,3,2315,Southampton,...,Miss,Elizabeth Gladys 'Millvina',2/2/1912,NaN,5/31/2009,2/2/1912,C,0.2,1.0,2.0
1,498,"DANBOM, Master Gilbert Sigvard Emanuel",0,1,male,0.33,3rd Class Passenger,3,347080,Southampton,...,Master,Gilbert Sigvard Emanuel,11/16/1911,NaN,4/15/1912,11/16/1911,C,0.0,0.0,2.0
2,1961,"TANNuS, Master As'ad",0,1,male,0.42,3rd Class Passenger,3,2625,Cherbourg,...,Master,As'ad,11/8/1911,NaN,6/12/1931,11/8/1911,C,0.0,0.0,1.0
3,811,"HäMäLäINEN, Master Viljo Unto Johannes",0,1,male,0.58,2nd Class Passenger,2,250649,Southampton,...,Master,Viljo Unto Johannes,8/29/1911,NaN,,8/29/1911,C,1.0,1.0,1.0
4,1551,"PEACOCK, Master Albert Edward",0,1,male,0.58,3rd Class Passenger,3,3101315,Southampton,...,Master,Albert Edward,9/8/1911,NaN,4/15/1912,9/8/1911,C,1.0,1.0,1.0



--- INFORMACION GENERAL DE TIPOS (df.info) ---
<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Name_ID      2208 non-null   int64  
 1   Name         2208 non-null   str    
 2   Female       2208 non-null   int64  
 3   Male         2208 non-null   int64  
 4   Sex          2208 non-null   str    
 5   Age          2154 non-null   float64
 6   Class/Dept   2208 non-null   str    
 7   Class        2208 non-null   str    
 8   Ticket       1317 non-null   str    
 9   Joined       2208 non-null   str    
 10  Occupation   1587 non-null   str    
 11  Boat [Body]  2208 non-null   str    
 12  Price        2208 non-null   str    
 13  Job          958 non-null    str    
 14  Survived     2208 non-null   int64  
 15  URL          2208 non-null   str    
 16  Last         2208 non-null   str    
 17  Title        2208 non-null   str    
 18  First      

,Valores Nulos,Porcentaje (%)
Age,54,2.45
Ticket,891,40.35
Occupation,621,28.12
Job,1250,56.61
DoB,977,44.25
Year_Birth,1587,71.88
DoB_Clean,355,16.08
Age_F,2,0.09
sibsp,899,40.72
parch,899,40.72


In [3]:
# Resumen descriptivo de numericas y tablas de frecuencias de variables categoricas clave
print('--- ESTADISTICAS DESCRIPTIVAS DE VARIABLES NUMERICAS ---')
display(df.describe().T)

print('\n--- DISTRIBUCION DE FRECUENCIAS: VARIABLE OBJETIVO (Survived) ---')
display(df['Survived'].value_counts().to_frame(name='Conteo').assign(
    Proporcion=df['Survived'].value_counts(normalize=True).round(4)
))

print('\n--- DISTRIBUCION DE FRECUENCIAS: ROL FUNCIONAL (Class/Dept) ---')
display(df['Class/Dept'].value_counts().to_frame(name='Conteo').assign(
    Proporcion=df['Class/Dept'].value_counts(normalize=True).round(4)
))

print('\n--- DISTRIBUCION DE FRECUENCIAS: PUERTO DE EMBARQUE (Joined) ---')
display(df['Joined'].value_counts(dropna=False).to_frame(name='Conteo').assign(
    Proporcion=df['Joined'].value_counts(normalize=True, dropna=False).round(4)
))

--- ESTADISTICAS DESCRIPTIVAS DE VARIABLES NUMERICAS ---


,count,mean,std,min,25%,50%,75%,max
Name_ID,2208.0,1104.500000,637.539018,1.00,552.75,1104.5,1656.25,2208.0
Female,2208.0,0.221467,0.415328,0.00,0.00,0.0,0.00,1.0
Male,2208.0,0.778533,0.415328,0.00,1.00,1.0,1.00,1.0
Age,2154.0,30.158770,11.958248,0.17,22.00,29.0,37.00,71.0
Survived,2208.0,0.322464,0.467525,0.00,0.00,0.0,1.00,1.0
Age_F,2206.0,30.146963,12.114540,0.00,22.00,29.0,38.00,74.0
sibsp,1309.0,0.498854,1.041658,0.00,0.00,0.0,1.00,8.0
parch,1309.0,0.385027,0.865560,0.00,0.00,0.0,0.00,9.0



--- DISTRIBUCION DE FRECUENCIAS: VARIABLE OBJETIVO (Survived) ---


,Conteo,Proporcion
Survived,,
0,1496,0.6775
1,712,0.3225



--- DISTRIBUCION DE FRECUENCIAS: ROL FUNCIONAL (Class/Dept) ---


,Conteo,Proporcion
Class/Dept,,
3rd Class Passenger,709,0.3211
Victualling Crew,431,0.1952
Engineering Crew,325,0.1472
1st Class Passenger,324,0.1467
2nd Class Passenger,276,0.1250
Restaurant Staff,69,0.0312
Deck Crew,66,0.0299
Band,8,0.0036



--- DISTRIBUCION DE FRECUENCIAS: PUERTO DE EMBARQUE (Joined) ---


,Conteo,Proporcion
Joined,,
Southampton,1614,0.7310
Cherbourg,272,0.1232
Belfast,199,0.0901
Queenstown,123,0.0557


**Observaciones del primer barrido:**

1. **Magnitud censal y desbalance del target:** El dataset contiene exactamente 2,208 registros, lo cual representa el censo exhaustivo de personas a bordo (no solo una muestra civil). La tasa general de supervivencia es del 32.25% (712 sobrevivientes) frente al 67.75% de mortalidad (1,496 fallecidos). Este desbalance exige que cualquier partición de validación sea estrictamente estratificada y que el modelo evaluador supere métricas más exigentes que el accuracy bruto.
2. **Estructura jerárquica y operativa (`Class/Dept`):** A diferencia de los datasets estándar del Titanic que solo contemplan 1ª, 2ª y 3ª clase, este censo incorpora departamentos funcionales de tripulación: personal de aprovisionamiento (`Victualling Crew`: 431 personas, 19.52%), maquinistas y fogoneros (`Engineering Crew`: 325 personas, 14.72%), personal de cubierta (`Deck Crew`: 66 personas, 2.99%), restaurante a la carta (`Restaurant Staff`: 69 personas, 3.13%) y la banda de música (`Band`: 8 personas). La 3ª clase es el grupo civil mayoritario con 709 pasajeros (32.11%).
3. **Presencia de variables post-evento y calidad de registros familiares:** Se detecta la presencia de columnas como `Boat [Body]`, `Date_Death` y `URL`. Estas columnas contienen información retrospectiva directa del desenlace (fuga de target o *data leakage*). Asimismo, variables demográficas como `sibsp` y `parch` exhiben 899 valores ausentes que coinciden exactamente con la tripulación, lo que indica un mecanismo sistemático condicionado al rol (MAR) que no debe descartarse con `dropna()`, sino imputarse coherentemente.

## 4. Calidad de datos

*Faltantes (y su mecanismo: MCAR, MAR o MNAR, con argumento), duplicados y outliers. Para cada problema: qué hicieron y por qué.*

## 5. Análisis exploratorio

*Distribuciones, relaciones y la variable objetivo. Cada gráfica lleva debajo una conclusión escrita.*

## 6. Feature engineering

*Encodings, escalamiento y variables derivadas. Partición train/test antes de ajustar cualquier transformación.*

## 7. Modelo baseline

*Baseline trivial frente al modelo k-NN.*

## 8. Conclusiones y límites

*Respuesta fundamentada y límites del análisis.*

## 9. Bitácora de colaboración con AI

*Documentación de asistencia.*

## Anexo: peer critique (se llena en clase)

**Revisor:**

1. ¿Entendí la pregunta del proyecto sin preguntarle al autor?
2. ¿Hay alguna gráfica o celda que yo borraría? ¿Cuál y por qué?
3. ¿Veo algún punto donde se pudo haber colado información del test?